In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier

In [2]:
train_part1 = pd.read_parquet("../data/test.parquet", engine='fastparquet')
event_id = train_part1["event_id"]

In [3]:
train_part1.info()

<class 'pandas.DataFrame'>
RangeIndex: 633683 entries, 0 to 633682
Data columns (total 23 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   customer_id                 633683 non-null  int64  
 1   event_id                    633683 non-null  int64  
 2   event_dttm                  633683 non-null  object 
 3   event_type_nm               633683 non-null  int32  
 4   event_desc                  633683 non-null  int32  
 5   channel_indicator_type      633683 non-null  int32  
 6   channel_indicator_sub_type  633683 non-null  int32  
 7   operaton_amt                293202 non-null  float64
 8   currency_iso_cd             301466 non-null  float64
 9   mcc_code                    154227 non-null  object 
 10  pos_cd                      42205 non-null   float64
 11  accept_language             63501 non-null   object 
 12  browser_language            53907 non-null   object 
 13  timezone                 

In [4]:
def most_popular(dataframe):            
    cols = dataframe.columns
    for i in cols:
        print(f"{i}:")
        print(f"{dataframe[i].mode(dropna=False)}\n")

In [5]:
delete = ["accept_language", "browser_language", "timezone", "session_id", "operating_system_type", \
        "battery", "device_system_version", "screen_size"]

train_part1.drop(columns=delete, inplace=True)

In [6]:
#########
train_part1.fillna(value=-1, inplace=True)
train_part1["phone_voip_call_state"] = train_part1["phone_voip_call_state"].astype(dtype="int32")
train_part1["web_rdp_connection"] = train_part1["web_rdp_connection"].astype(dtype="int32")

In [7]:
train_part1["Hour"] = pd.to_datetime(train_part1["event_dttm"]).dt.hour
train_part1.drop(columns="event_dttm", inplace=True)

In [8]:
train_part1.drop(columns=["customer_id", "event_id"], inplace=True)

In [9]:
train_part1

,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,Hour
0,8,122,2,0,-1.0,-1.0,-1,-1.0,-1,-1,-1,-1,14
1,14,42,4,15,247750.0,0.0,-1,-1.0,0,0,-1,0,14
2,7,56,4,15,-1.0,-1.0,-1,-1.0,0,0,-1,0,14
3,7,56,4,15,-1.0,-1.0,-1,-1.0,0,0,-1,0,14
4,8,86,4,15,496350.0,0.0,-1,-1.0,0,0,-1,0,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...
633678,9,105,4,14,13965.0,0.0,-1,-1.0,-1,-1,-1,-1,12
633679,7,56,4,15,-1.0,-1.0,-1,-1.0,0,0,-1,0,16
633680,7,56,4,15,-1.0,-1.0,-1,-1.0,0,0,-1,0,16
633681,7,56,4,15,-1.0,-1.0,-1,-1.0,0,0,-1,0,2


In [10]:
train_part1["mcc_code"] = train_part1["mcc_code"].astype(dtype="int32")
train_part1["compromised"] = train_part1["compromised"].astype(dtype="int32")
train_part1["developer_tools"] = train_part1["developer_tools"].astype(dtype="int32")

In [11]:
train_part1.info()

<class 'pandas.DataFrame'>
RangeIndex: 633683 entries, 0 to 633682
Data columns (total 13 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   event_type_nm               633683 non-null  int32  
 1   event_desc                  633683 non-null  int32  
 2   channel_indicator_type      633683 non-null  int32  
 3   channel_indicator_sub_type  633683 non-null  int32  
 4   operaton_amt                633683 non-null  float64
 5   currency_iso_cd             633683 non-null  float64
 6   mcc_code                    633683 non-null  int32  
 7   pos_cd                      633683 non-null  float64
 8   developer_tools             633683 non-null  int32  
 9   phone_voip_call_state       633683 non-null  int32  
 10  web_rdp_connection          633683 non-null  int32  
 11  compromised                 633683 non-null  int32  
 12  Hour                        633683 non-null  int32  
dtypes: float64(3), int32(10)


Получили обработанный набор данных, теперь можно переходить прогнозированию.

In [12]:
model = CatBoostClassifier()
model.load_model('../Models/Model_1.cbm')

CatBoostClassifier(class_names=[0, 1], class_weights=[1, 500], depth=4, iterations=200, loss_function='Logloss', verbose=0)

In [13]:
predict = model.predict(train_part1)
y_pred_proba = model.predict_proba(train_part1)[:, 1]

In [14]:
y_pred_proba = pd.Series(y_pred_proba, name="predict")
y_pred_proba

0         0.007281
1         0.120822
2         0.158721
3         0.158721
4         0.082661
            ...   
633678    0.069048
633679    0.156890
633680    0.156890
633681    0.139369
633682    0.139369
Name: predict, Length: 633683, dtype: float64

In [15]:
event_id

0         123707242230467
1         123234793229123
2         125837545055907
3         126456020999239
4         125090221587312
               ...       
633678    125528306953183
633679    123690063072824
633680    125734465101827
633681    125193299828684
633682    125657156303168
Name: event_id, Length: 633683, dtype: int64

In [16]:
submit = pd.concat([event_id, y_pred_proba], names=["event_id", "predict"], axis=1)

In [17]:
submit

,event_id,predict
0,123707242230467,0.007281
1,123234793229123,0.120822
2,125837545055907,0.158721
3,126456020999239,0.158721
4,125090221587312,0.082661
...,...,...
633678,125528306953183,0.069048
633679,123690063072824,0.156890
633680,125734465101827,0.156890
633681,125193299828684,0.139369


In [18]:
submit.to_csv("submit.csv", index=False)